### CVAT Custom Dataset

Uses the manually annotated dataset exported from CVAT (`data/cvat_batch_1/`).

This notebook is separate from Notebooks 3 and 4 so it can be re-run any time new labels are added in CVAT. It always reads whatever is currently on disk.

Workflow:
1. Inspect the export (images, labels, classes).
2. Filter out images with no labels.
3. Split into train / val (80/20).
4. Write a `data.yaml` for Ultralytics.
5. Train YOLO11n.
6. Evaluate on the val split.
7. Save annotated examples.

Outputs: filtered dataset, split dataset, training run, evaluation metrics, annotated images.

In [10]:
from pathlib import Path
import shutil
import random
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.14.1+cpu
CUDA available: False


### Paths

Input: `data/cvat_batch_1/` — raw CVAT export (YOLO 1.0 format).
Intermediate: `data/cvat_batch_1_filtered/` — only labelled images.
Split: `data/cvat_batch_1_split/` — train / valid folders plus `data.yaml`.
Results: `results/cvat_batch_1/`.

In [20]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = ROOT / "data" / "cvat_batch_1"
FILTERED_DIR = ROOT / "data" / "cvat_batch_1_filtered"
SPLIT_DIR = ROOT / "data" / "cvat_batch_1_split"
RESULTS_DIR = ROOT / "results" / "cvat_batch_1"
RUNS_DIR = RESULTS_DIR / "training"
EVAL_DIR = RESULTS_DIR / "evaluation"

for d in [RESULTS_DIR, RUNS_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = 0 if torch.cuda.is_available() else "cpu"

print(f"Raw CVAT export: {RAW_DIR.relative_to(ROOT)}  exists={RAW_DIR.exists()}")
print(f"Results:         {RESULTS_DIR.relative_to(ROOT)}")
print(f"Device:          {device}")

Raw CVAT export: data\cvat_batch_1  exists=True
Results:         results\cvat_batch_1
Device:          cpu


In [21]:
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

all_images = sorted(
    p for p in RAW_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in EXTS
)

all_labels = {p.stem: p for p in RAW_DIR.rglob("*.txt")}

print(f"Raw folder: {RAW_DIR}")
print(f"Images found: {len(all_images)}")
print(f"Labels found: {len(all_labels)}")

if all_images:
    print("\nFirst 3 images:")
    for p in all_images[:3]:
        print("  ", p.relative_to(RAW_DIR))

if all_labels:
    sample_key = next(iter(all_labels))
    print(f"\nSample label {sample_key}.txt:")
    print(all_labels[sample_key].read_text()[:200])

Raw folder: d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data\cvat_batch_1
Images found: 321
Labels found: 129

First 3 images:
   images\train\20250326_110923.jpg
   images\train\20250326_112218.jpg
   images\train\20250326_130133.jpg

Sample label train.txt:
data/images/train/20250326_110923.jpg
data/images/train/20250326_112218.jpg
data/images/train/20250326_130133.jpg
data/images/train/20250326_130204.jpg
data/images/train/20250326_130347.jpg
data/image


### Filter to labelled images only

In [22]:
if FILTERED_DIR.exists():
    shutil.rmtree(FILTERED_DIR)
(FILTERED_DIR / "images").mkdir(parents=True, exist_ok=True)
(FILTERED_DIR / "labels").mkdir(parents=True, exist_ok=True)

kept, skipped_no_label, skipped_empty = 0, 0, 0

for img in all_images:
    lbl = all_labels.get(img.stem)

    if lbl is None:
        skipped_no_label += 1
        continue

    if lbl.stat().st_size == 0:
        skipped_empty += 1
        continue

    shutil.copy2(img, FILTERED_DIR / "images" / img.name)
    shutil.copy2(lbl, FILTERED_DIR / "labels" / lbl.name)
    kept += 1

print(f"Kept:               {kept} images with non-empty labels")
print(f"Skipped (no label): {skipped_no_label}")
print(f"Skipped (empty):    {skipped_empty}")
print(f"Output:             {FILTERED_DIR.relative_to(ROOT)}")

Kept:               128 images with non-empty labels
Skipped (no label): 193
Skipped (empty):    0
Output:             data\cvat_batch_1_filtered


### Split into train / valid

80/20 split

In [24]:
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

for split in ["train", "valid"]:
    (SPLIT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (SPLIT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

images = sorted((FILTERED_DIR / "images").glob("*"))
random.seed(0)
random.shuffle(images)

VAL_RATIO = 0.2
n_val = max(1, int(len(images) * VAL_RATIO)) if images else 0
val_names = set(p.name for p in images[:n_val])

for img in images:
    split = "valid" if img.name in val_names else "train"
    lbl = FILTERED_DIR / "labels" / f"{img.stem}.txt"
    shutil.copy2(img, SPLIT_DIR / split / "images" / img.name)
    shutil.copy2(lbl, SPLIT_DIR / split / "labels" / lbl.name)

n_train = len(images) - n_val
print(f"Train: {n_train}")
print(f"Valid: {n_val}")
print(f"Total: {len(images)}")

Train: 103
Valid: 25
Total: 128


### Write data.yaml

In [25]:
CLASS_NAMES = ["gas_cylinder"]

if yaml_files:
    raw_cfg = yaml.safe_load(yaml_files[0].read_text())
    if raw_cfg.get("names"):
        names = raw_cfg["names"]
        if isinstance(names, dict):
            CLASS_NAMES = [names[k] for k in sorted(names)]
        else:
            CLASS_NAMES = list(names)
        print(f"Using class names from CVAT export: {CLASS_NAMES}")
    else:
        print(f"No names found in CVAT YAML. Using default: {CLASS_NAMES}")
else:
    print(f"No YAML found in CVAT export. Using default: {CLASS_NAMES}")

yaml_content = {
    "path": str(SPLIT_DIR),
    "train": "train/images",
    "val": "valid/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = SPLIT_DIR / "data.yaml"
yaml_path.write_text(yaml.safe_dump(yaml_content, sort_keys=False))

print(f"\nWrote {yaml_path.relative_to(ROOT)}")
print(yaml_path.read_text())

Using class names from CVAT export: ['gas-bottles']

Wrote data\cvat_batch_1_split\data.yaml
path: d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data\cvat_batch_1_split
train: train/images
val: valid/images
nc: 1
names:
- gas-bottles



### Sanity check

Confirm the split folders and YAML agree before training.

In [26]:
def count(path, pattern="*"):
    return sum(1 for p in path.glob(pattern) if p.is_file())

for split in ["train", "valid"]:
    imgs = count(SPLIT_DIR / split / "images")
    lbls = count(SPLIT_DIR / split / "labels", "*.txt")
    print(f"{split:6s}  images={imgs:4d}  labels={lbls:4d}")

assert (SPLIT_DIR / "train" / "images").exists(), "train/images missing"
assert (SPLIT_DIR / "valid" / "images").exists(), "valid/images missing"

print(f"\nYAML: {yaml_path.relative_to(ROOT)}")

train   images= 103  labels= 103
valid   images=  25  labels=  25

YAML: data\cvat_batch_1_split\data.yaml


### Training settings

In [27]:
EPOCHS = 20
IMGSZ = 416
BATCH = 4
WORKERS = 0
SEED = 0
RUN_NAME = "yolo11n_cvat_batch_1"

print(f"epochs={EPOCHS}  imgsz={IMGSZ}  batch={BATCH}  workers={WORKERS}")

epochs=20  imgsz=416  batch=4  workers=0


### Train

Trains from COCO weights.

In [29]:
model = YOLO("yolo11n.pt")

model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    device=device,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=True,
    val=True,
    seed=SEED,
)

RUN_DIR = RUNS_DIR / RUN_NAME
print(f"\nRun directory: {RUN_DIR.relative_to(ROOT)}")

New https://pypi.org/project/ultralytics/8.4.175 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.174  Python-3.11.9 torch-2.14.1+cpu CPU (AMD Ryzen 7 6800HS with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data\cvat_batch_1_split\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01

### Read training metrics

Last-epoch metrics from the run's `results.csv`.

In [30]:
def pick(row, *keys, default=None):
    for k in keys:
        for c in row.index:
            if c.lower() == k.lower():
                return row[c]
    return default

results_csv = RUN_DIR / "results.csv"
df_train = pd.read_csv(results_csv)
df_train.columns = [c.strip() for c in df_train.columns]
last = df_train.iloc[-1]

train_row = {
    "dataset": "cvat_batch_1",
    "split": "train",
    "images_train": count(SPLIT_DIR / "train" / "images"),
    "images_valid": count(SPLIT_DIR / "valid" / "images"),
    "classes": str(CLASS_NAMES),
    "precision": pick(last, "metrics/precision(B)", "precision"),
    "recall": pick(last, "metrics/recall(B)", "recall"),
    "mAP50": pick(last, "metrics/mAP50(B)", "mAP50"),
    "mAP50-95": pick(last, "metrics/mAP50-95(B)", "mAP50-95"),
}

print("=" * 60)
print("TRAINING SUMMARY — CVAT BATCH 1")
print("=" * 60)
for k, v in train_row.items():
    print(f"{k:20s}: {v}")
print("=" * 60)

SUMMARY_CSV = RUN_DIR / "training_summary.csv"
pd.DataFrame([train_row]).to_csv(SUMMARY_CSV, index=False)
print(f"Saved to {SUMMARY_CSV.relative_to(ROOT)}")

TRAINING SUMMARY — CVAT BATCH 1
dataset             : cvat_batch_1
split               : train
images_train        : 103
images_valid        : 25
classes             : ['gas-bottles']
precision           : 0.99456
recall              : 0.97222
mAP50               : 0.975
mAP50-95            : 0.88264
Saved to results\cvat_batch_1\training\yolo11n_cvat_batch_1\training_summary.csv


### Standalone evaluation on the val split

Loads `best.pt` and runs `model.val()` once.

In [31]:
BEST_PT = RUN_DIR / "weights" / "best.pt"
assert BEST_PT.exists(), f"best.pt missing: {BEST_PT}"

eval_model = YOLO(str(BEST_PT))

val_metrics = eval_model.val(
    data=str(yaml_path),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    project=str(EVAL_DIR),
    name="val",
    exist_ok=True,
    plots=True,
    verbose=False,
)

val_row = {
    "dataset": "cvat_batch_1",
    "split": "val",
    "precision": float(val_metrics.box.mp),
    "recall": float(val_metrics.box.mr),
    "mAP50": float(val_metrics.box.map50),
    "mAP50-95": float(val_metrics.box.map),
}

print("=" * 60)
print("STANDALONE VALIDATION — CVAT BATCH 1")
print("=" * 60)
for k, v in val_row.items():
    print(f"{k:20s}: {v}")
print("=" * 60)

EVAL_CSV = RESULTS_DIR / "evaluation_metrics.csv"
pd.DataFrame([train_row, val_row]).to_csv(EVAL_CSV, index=False)
print(f"Saved to {EVAL_CSV.relative_to(ROOT)}")

Ultralytics 8.4.174  Python-3.11.9 torch-2.14.1+cpu CPU (AMD Ryzen 7 6800HS with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 656.2220.5 MB/s, size: 3496.8 KB)
val: Scanning D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data\cvat_batch_1_split\valid\labels.cache... 25 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 25/25  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.7it/s 2.6s0.5s
                   all         25         36      0.995      0.972      0.975      0.877
Speed: 0.3ms preprocess, 20.4ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\results\cvat_batch_1\evaluation\val
STANDALONE VALIDATION — CVAT BATCH 1
dataset             : cvat_batch_1
split               : val
precision           : 0.99453820957279

### Annotated examples

Runs the fine-tuned model on a few val images and saves the annotated versions.

In [32]:
import cv2

ANNOTATED_DIR = RESULTS_DIR / "annotated"
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

val_images_dir = SPLIT_DIR / "valid" / "images"
val_images = sorted(p for p in val_images_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
sample = val_images[:5]

for img_path in sample:
    res = eval_model.predict(
        source=str(img_path),
        conf=0.25,
        imgsz=IMGSZ,
        device=device,
        verbose=False,
    )[0]

    annotated = res.plot()
    out_path = ANNOTATED_DIR / f"pred_{img_path.stem}.jpg"
    cv2.imwrite(str(out_path), annotated)

    n = 0 if res.boxes is None else len(res.boxes)
    print(f"{img_path.name:35s} detections={n}")

print(f"\nAnnotated images saved to {ANNOTATED_DIR.relative_to(ROOT)}")

20250326_130204.jpg                 detections=1
20250326_130347.jpg                 detections=1
20250326_134806.jpg                 detections=1
20250326_143708.jpg                 detections=2
20250326_143754.jpg                 detections=1

Annotated images saved to results\cvat_batch_1\annotated


In [33]:
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SPLIT = ROOT / "data" / "cvat_batch_1_split"

def scene_prefix(name):
    for p in ["frame_14_55_back_left", "frame_14_55_back_right", "frame_14_55_front", "20250326"]:
        if name.startswith(p):
            return p
    return "other"

for split in ["train", "valid"]:
    files = sorted((SPLIT / split / "images").glob("*"))
    counts = Counter(scene_prefix(f.name) for f in files)
    print(f"{split}: {dict(counts)}")

train: {'20250326': 73, 'frame_14_55_back_left': 18, 'frame_14_55_back_right': 12}
valid: {'20250326': 16, 'frame_14_55_back_left': 7, 'frame_14_55_back_right': 2}
